# 🌴 Taller LiDAR Forestal · Cuaderno 1
## Del CHM al conteo de árboles: segmentación de copas con *Watershed*

**Kaab Map · Sesión final del taller (≈ 40 min)**

En este cuaderno vamos a convertir un **Modelo de Altura de Dosel (CHM)** en un **inventario de árboles**: cuántos hay, dónde están, qué altura tienen y qué tamaño de copa ocupan.

No necesitas programar: casi todo el código está oculto en celdas de formulario (ícono ▶️). Tu trabajo es **entender cada paso, mover los parámetros y discutir los resultados**.

| Paso | ¿Qué hacemos? | Pregunta forestal que responde |
|---|---|---|
| 1 | Repaso: CHM = DSM − DTM | ¿Qué altura tiene la vegetación? |
| 2 | Intuición: el CHM como relieve invertido | ¿Por qué funciona *watershed*? |
| 3 | Suavizado gaussiano | ¿Cuánto ruido del dosel ignoramos? |
| 4 | Detección de cimas (máximos locales) | ¿Dónde está cada árbol? |
| 5 | Watershed con marcadores | ¿Qué área ocupa cada copa? |
| 6 | Atributos por árbol y métricas de rodal | Densidad, altura, cobertura |
| 7 | Validación y caso difícil | ¿Qué tan confiable es el conteo? |

> 💡 **Cómo usar este cuaderno:** ejecuta las celdas en orden con **Shift + Enter**. Cuando veas ✏️ hay un ejercicio; cuando veas 🔎 hay una pregunta para discutir en grupo.

In [ ]:
#@title ⚙️ Paso 0 · Preparar el entorno y descargar los datos (ejecuta y espera ~1 min)
#@markdown Si el instructor te compartió un **ID de Google Drive** del archivo `datos_taller.zip`, pégalo aquí.
#@markdown Si lo dejas vacío, se te pedirá subir el ZIP desde tu computadora.
ID_ZIP_DRIVE = ""  #@param {type:"string"}

import os, sys, subprocess, zipfile, warnings
warnings.filterwarnings("ignore")
EN_COLAB = "google.colab" in sys.modules

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    import rasterio
except ImportError:
    _pip("rasterio")

if not os.path.exists("datos/zona1_plantacion_chm.tif"):
    if ID_ZIP_DRIVE.strip():
        _pip("gdown")
        import gdown
        gdown.download(id=ID_ZIP_DRIVE.strip(), output="datos_taller.zip", quiet=False)
    elif EN_COLAB:
        from google.colab import files
        print("Selecciona el archivo datos_taller.zip ...")
        subido = files.upload()
        os.rename(list(subido.keys())[0], "datos_taller.zip")
    with zipfile.ZipFile("datos_taller.zip") as z:
        z.extractall(".")

print("Archivos disponibles en la carpeta datos/:")
for f in sorted(os.listdir("datos")):
    print(f"  {f:32s} {os.path.getsize('datos/' + f) / 1e6:5.1f} MB")

In [ ]:
#@title 🧰 Funciones de apoyo (no necesitas leerlas, solo ejecútala)
import numpy as np, pandas as pd, geopandas as gpd, rasterio
import matplotlib.pyplot as plt
from rasterio.features import shapes
from shapely.geometry import shape, Point
from skimage.filters import gaussian
from skimage.feature import peak_local_max
from skimage.segmentation import watershed, find_boundaries
from skimage.measure import regionprops, profile_line
plt.rcParams.update({"figure.dpi": 100, "font.size": 10, "axes.titlesize": 11})


def cargar_zona(nombre):
    """Lee el CHM y la ortofoto de una zona. Los ejes de las figuras van en metros desde la esquina."""
    with rasterio.open(f"datos/{nombre}_chm.tif") as src:
        chm = src.read(1).astype("float32")
        tr, crs = src.transform, src.crs
    with rasterio.open(f"datos/{nombre}_orto.tif") as src:
        rgb = np.moveaxis(src.read([1, 2, 3]), 0, -1)
    res = tr.a
    alto, ancho = chm.shape
    return dict(nombre=nombre, chm=chm, rgb=rgb, transform=tr, crs=crs, res=res,
                ext=[0, ancho * res, alto * res, 0], area_ha=alto * ancho * res * res / 1e4)


def segmentar(zona, sigma_m, dist_min_m, altura_min_m):
    """Las tres líneas clave del método: suavizar -> cimas -> watershed."""
    res = zona["res"]
    suave = gaussian(zona["chm"], sigma=sigma_m / res, preserve_range=True)
    cimas = peak_local_max(suave, min_distance=max(1, int(dist_min_m / res)),
                           threshold_abs=altura_min_m, exclude_border=False)
    marcadores = np.zeros(suave.shape, dtype=np.int32)
    marcadores[cimas[:, 0], cimas[:, 1]] = np.arange(1, len(cimas) + 1)
    copas = watershed(-suave, marcadores, mask=suave > altura_min_m)
    return suave, cimas, copas


def tabla_arboles(zona, copas, cimas):
    """Un renglón por árbol: posición de la cima y atributos de su copa."""
    res, tr = zona["res"], zona["transform"]
    filas = []
    for r in regionprops(copas, intensity_image=zona["chm"]):
        fila, col = cimas[r.label - 1]          # la copa k nació del marcador k
        area = r.area * res * res
        filas.append(dict(arbol_id=r.label,
                          x=round(tr.c + (col + 0.5) * res, 2), y=round(tr.f - (fila + 0.5) * res, 2),
                          x_m=(col + 0.5) * res, y_m=(fila + 0.5) * res,
                          altura_max_m=round(float(r.intensity_max), 2),
                          altura_media_m=round(float(r.intensity_mean), 2),
                          area_copa_m2=round(area, 2),
                          diam_copa_m=round(2 * np.sqrt(area / np.pi), 2)))
    return pd.DataFrame(filas)


def bordes_rgb(zona, copas, color=(255, 40, 40)):
    img = zona["rgb"].copy()
    img[find_boundaries(copas, mode="inner")] = color
    return img


def ventana(ax, zona, x0, y0, lado):
    ax.set_xlim(x0, x0 + lado); ax.set_ylim(y0 + lado, y0)


def a_geopackage(zona, copas, tabla, ruta):
    geoms = [(shape(g), int(v)) for g, v in shapes(copas, mask=copas > 0, transform=zona["transform"])]
    pol = gpd.GeoDataFrame({"arbol_id": [v for _, v in geoms]}, geometry=[g for g, _ in geoms], crs=zona["crs"])
    pol = pol.dissolve(by="arbol_id").reset_index().merge(tabla.drop(columns=["x_m", "y_m"]), on="arbol_id")
    pts = gpd.GeoDataFrame(tabla.drop(columns=["x_m", "y_m"]),
                           geometry=[Point(x, y) for x, y in zip(tabla.x, tabla.y)], crs=zona["crs"])
    pol.to_file(ruta, layer="copas", driver="GPKG")
    pts.to_file(ruta, layer="cimas", driver="GPKG")
    return pol, pts

print("Funciones listas ✔")

---
## Paso 1 · Repaso: ¿de dónde sale el CHM?

En los módulos anteriores generamos dos superficies a partir de la nube de puntos LiDAR:

* **DSM** (*Modelo Digital de Superficie*): elevación de lo **primero** que toca el láser (copas, techos).
* **DTM** (*Modelo Digital de Terreno*): elevación del **suelo**, interpolada con los puntos clasificados como terreno.

Si restamos el terreno a la superficie obtenemos la **altura de los objetos sobre el suelo**:

$$\text{CHM} = \text{DSM} - \text{DTM}$$

Trabajaremos con recortes de **150 × 150 m** (2.25 ha) a **10 cm por píxel** de la Etapa B:

* `zona1_plantacion`: plantación de palma en marco regular (caso "fácil").
* `zona2_natural`: vegetación natural ribereña (caso "difícil").

In [ ]:
# Leemos el DSM y el DTM de la zona 1 y calculamos el CHM nosotros mismos
with rasterio.open("datos/zona1_plantacion_dsm.tif") as src:
    dsm = src.read(1)
with rasterio.open("datos/zona1_plantacion_dtm.tif") as src:
    dtm = src.read(1)

chm_calculado = dsm - dtm          # <- ¡todo el "secreto" del CHM está en esta resta!

print(f"DSM: de {dsm.min():.1f} a {dsm.max():.1f} m s.n.m.")
print(f"DTM: de {dtm.min():.1f} a {dtm.max():.1f} m s.n.m.")
print(f"CHM: de {chm_calculado.min():.2f} a {chm_calculado.max():.2f} m sobre el suelo")

In [ ]:
#@title 🖼️ Ver DSM, DTM y CHM
z1 = cargar_zona("zona1_plantacion")
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for a, img, t, cm in [(ax[0], dsm, "DSM (superficie)", "terrain"), (ax[1], dtm, "DTM (terreno)", "terrain"),
                      (ax[2], np.clip(chm_calculado, 0, None), "CHM = DSM − DTM", "viridis")]:
    im = a.imshow(img, cmap=cm, extent=z1["ext"]); a.set_title(t); plt.colorbar(im, ax=a, shrink=.75, label="m")
ax[3].imshow(z1["rgb"], extent=z1["ext"]); ax[3].set_title("Ortofoto (misma zona)")
for a in ax: a.set_xlabel("m")
plt.tight_layout(); plt.show()

✏️ **Ejercicio 1** (2 min)
1. ¿Cuál es la altura máxima de la vegetación en la zona? ¿Es razonable para una palma?
2. El CHM tiene algunos valores **negativos**. ¿Qué significan físicamente? ¿Pueden existir árboles con altura negativa? (Pista: errores de interpolación del DTM y ruido del sensor). Por eso los recortamos a 0:

In [ ]:
print("Píxeles con altura negativa:", int((chm_calculado < 0).sum()))
chm_calculado = np.clip(chm_calculado, 0, None)   # todo lo negativo pasa a 0 m

---
## Paso 2 · Intuición: el CHM como un relieve… invertido

*Watershed* significa **"parteaguas"** o **"línea divisoria de cuencas"**. El algoritmo viene de la hidrología y de la morfología matemática (Beucher y Lantuéjoul, 1979).

Imagina que el CHM es un paisaje:

1. Cada **copa** es una **montaña** y la **cima** es el ápice del árbol.
2. Si **invertimos** el CHM (lo multiplicamos por −1), las montañas se vuelven **cuencas** (hoyos).
3. Empezamos a **inundar** desde el fondo de cada hoyo. Cada hoyo se llena con agua de un color distinto.
4. Cuando el agua de dos cuencas está por tocarse, construimos un **dique**. Esos diques son los **límites entre copas**.

Veámoslo primero en 3D y luego en un **perfil** (transecto) que cruza una hilera de palmas.

In [ ]:
#@title 🏔️ Vista 3D: copas como montañas → copas como cuencas
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
sub = gaussian(z1["chm"], sigma=4, preserve_range=True)[700:1000:3, 700:1000:3]
yy, xx = np.mgrid[0:sub.shape[0], 0:sub.shape[1]] * 0.3
fig = plt.figure(figsize=(16, 6))
for i, (sup, t, cm) in enumerate([(sub, "CHM: cada copa es una montaña", "viridis"),
                                  (-sub, "−CHM: cada copa es una cuenca que se inunda", "Blues_r")]):
    a = fig.add_subplot(1, 2, i + 1, projection="3d")
    a.plot_surface(xx, yy, sup, cmap=cm, linewidth=0, antialiased=False)
    a.set_title(t); a.set_xlabel("m"); a.set_ylabel("m"); a.set_zlabel("altura (m)"); a.view_init(35, -60)
plt.tight_layout(); plt.show()

In [ ]:
#@title 🌊 Perfil: así "inunda" el algoritmo
A, B = (525, 117), (513, 802)                 # extremos del transecto (fila, columna)
perfil = profile_line(z1["chm"], A, B, linewidth=3)
perfil_s = gaussian(perfil, sigma=4, preserve_range=True)
dist = np.arange(len(perfil)) * z1["res"]
picos = peak_local_max(perfil_s, min_distance=30, threshold_abs=2)[:, 0]
m1d = np.zeros(len(perfil), int); m1d[picos] = np.arange(1, len(picos) + 1)
cuencas = watershed(-perfil_s, m1d)
diques = np.where(np.diff(cuencas) != 0)[0]
colores = plt.cm.tab10(np.arange(10))

fig, ax = plt.subplots(2, 2, figsize=(18, 8))
ax[0, 0].imshow(z1["rgb"], extent=z1["ext"]); ax[0, 0].plot([A[1]*.1, B[1]*.1], [A[0]*.1, B[0]*.1], "r-", lw=3)
ventana(ax[0, 0], z1, 5, 25, 80); ax[0, 0].set_title("Transecto sobre la ortofoto")
ax[0, 1].plot(dist, perfil, color="0.6", lw=1, label="CHM original")
ax[0, 1].plot(dist, perfil_s, color="green", lw=2.5, label="CHM suavizado")
ax[0, 1].plot(dist[picos], perfil_s[picos], "rv", ms=12, label="cimas = árboles")
ax[0, 1].set_title("① Perfil de alturas: cada montaña es una copa"); ax[0, 1].set_ylabel("altura (m)"); ax[0, 1].legend()
for k, nivel in enumerate([-7.5, -4.0]):
    a = ax[1, k]
    a.plot(dist, -perfil_s, "k", lw=2)
    agua = np.minimum(nivel, 0)
    for c in range(1, len(picos) + 1):
        sel = (cuencas == c) & (-perfil_s < agua)
        a.fill_between(dist, -perfil_s, agua, where=sel, color=colores[(c - 1) % 10], alpha=.6)
    if k == 1:
        for d in diques: a.axvline(dist[d], color="red", lw=2.5)
    a.axhline(nivel, color="steelblue", ls="--", lw=1)
    a.set_title(["② Invertimos y el agua empieza a subir desde cada cima",
                 "③ Donde dos cuencas se tocan → dique = límite entre copas"][k])
    a.set_xlabel("distancia sobre el transecto (m)"); a.set_ylabel("−altura (m)")
plt.tight_layout(); plt.show()
print(f"En este transecto el algoritmo encontró {len(picos)} árboles.")

🔎 **Pregunta para discutir:** en el perfil "original" (gris) hay pequeños picos dentro de una misma copa (hojas, frondas). ¿Qué pasaría si cada uno de esos picos se convirtiera en una cuenca? → **sobresegmentación**: un árbol partido en muchos. Por eso el primer paso es **suavizar**.

---
## Paso 3 · Suavizado gaussiano

Un filtro gaussiano reemplaza cada píxel por un **promedio ponderado** de sus vecinos; los vecinos cercanos pesan más que los lejanos. El parámetro **σ (sigma)** controla qué tan "ancha" es la campana:

* σ pequeño → conserva detalles (y ruido) → **muchas** cimas falsas.
* σ grande → aplana copas pequeñas y fusiona copas vecinas → **pocas** cimas.

Regla práctica: σ ≈ 5–15 % del diámetro de copa típico. Una palma tiene ~7 m de copa, así que probamos **σ ≈ 0.4 m**.

✏️ **Ejercicio 2:** mueve el deslizador `SIGMA_M`, vuelve a ejecutar y observa cómo cambia el número de cimas.

In [ ]:
#@title 🎚️ Efecto del suavizado
SIGMA_M = 0.4  #@param {type:"slider", min:0.0, max:1.5, step:0.1}
suave = gaussian(z1["chm"], sigma=max(SIGMA_M, 1e-3) / z1["res"], preserve_range=True)
n_cimas = len(peak_local_max(suave, min_distance=10, threshold_abs=2, exclude_border=False))

fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))
ax[0].imshow(z1["chm"], cmap="viridis", vmin=0, vmax=10, extent=z1["ext"]); ax[0].set_title("CHM original")
ax[1].imshow(suave, cmap="viridis", vmin=0, vmax=10, extent=z1["ext"]); ax[1].set_title(f"CHM suavizado (σ = {SIGMA_M} m)")
for a in ax[:2]: ventana(a, z1, 60, 60, 30)
ax[2].plot(dist, perfil, color="0.6", label="original")
ax[2].plot(dist, gaussian(perfil, sigma=max(SIGMA_M, 1e-3) / z1["res"], preserve_range=True), "g", lw=2.5, label=f"σ = {SIGMA_M} m")
ax[2].set_title("Mismo transecto"); ax[2].set_xlabel("m"); ax[2].legend()
plt.tight_layout(); plt.show()
print(f"Con σ = {SIGMA_M} m se detectan {n_cimas} máximos locales en toda la zona (2.25 ha).")

| σ (m) | Cimas detectadas (anótalo) |
|---|---|
| 0.0 | |
| 0.2 | |
| 0.4 | |
| 1.0 | |

---
## Paso 4 · Detección de cimas (máximos locales = árboles)

Una **cima** es un píxel que es **el más alto dentro de una ventana** a su alrededor. Este enfoque se conoce como *Local Maxima Filter* (LMF) y es el método clásico para detectar árboles individuales en LiDAR (Popescu y Wynne, 2004).

Dos parámetros controlan qué cuenta como árbol:

* **`DIST_MIN_M`**: distancia mínima entre dos cimas. Debería relacionarse con el **espaciamiento** entre árboles. En una plantación con marco de ~8 m, una distancia de 3 m es segura.
* **`ALTURA_MIN_M`**: altura mínima para considerar un árbol (descarta pasto, arbustos, ruido). En el inventario forestal suele usarse 2 m.

Estas son, literalmente, las líneas de código del método:

In [ ]:
SIGMA_M      = 0.4   # suavizado (m)
DIST_MIN_M   = 3.0   # distancia mínima entre árboles (m)
ALTURA_MIN_M = 2.0   # altura mínima de un árbol (m)

suave = gaussian(z1["chm"], sigma=SIGMA_M / z1["res"], preserve_range=True)
cimas = peak_local_max(suave, min_distance=int(DIST_MIN_M / z1["res"]),
                       threshold_abs=ALTURA_MIN_M, exclude_border=False)

print(f"🌴 Árboles detectados en la zona 1: {len(cimas)}")

In [ ]:
#@title 🖼️ Ver las cimas sobre el CHM y la ortofoto
fig, ax = plt.subplots(1, 2, figsize=(18, 8))
ax[0].imshow(suave, cmap="viridis", vmin=0, extent=z1["ext"]); ax[0].set_title(f"CHM suavizado + {len(cimas)} cimas")
ax[1].imshow(z1["rgb"], extent=z1["ext"]); ax[1].set_title("Ortofoto + cimas (zoom 50 × 50 m)")
for a in ax:
    a.plot(cimas[:, 1] * z1["res"], cimas[:, 0] * z1["res"], "r+", ms=9, mew=2)
ventana(ax[1], z1, 50, 50, 50)
plt.tight_layout(); plt.show()

---
## Paso 5 · Watershed con marcadores: de cimas a copas

Ya sabemos **dónde** está cada árbol; ahora queremos su **copa**. Usamos *watershed controlado por marcadores* (*marker-controlled watershed*):

1. Cada cima recibe un número (1, 2, 3, …) → son los **marcadores** (los "manantiales" de agua).
2. Se inunda el **CHM invertido** (`-suave`) solo desde esos marcadores. Así evitamos la sobresegmentación: **solo hay tantas copas como cimas**.
3. Una **máscara** (`suave > ALTURA_MIN_M`) impide que el agua se desborde sobre el suelo desnudo.

In [ ]:
marcadores = np.zeros(suave.shape, dtype=np.int32)
marcadores[cimas[:, 0], cimas[:, 1]] = np.arange(1, len(cimas) + 1)    # 1 número por árbol

copas = watershed(-suave, marcadores, mask=suave > ALTURA_MIN_M)       # ← el algoritmo

print(f"Copas segmentadas: {copas.max()}")

In [ ]:
#@title 🖼️ Ver las copas segmentadas
rng = np.random.default_rng(0)
paleta = rng.random((copas.max() + 1, 3)); paleta[0] = 1
fig, ax = plt.subplots(1, 3, figsize=(20, 7))
ax[0].imshow(paleta[copas], extent=z1["ext"]); ax[0].set_title("Cada color = una copa (etiquetas)")
ax[1].imshow(bordes_rgb(z1, copas), extent=z1["ext"]); ax[1].set_title("Límites de copas sobre la ortofoto")
ax[2].imshow(bordes_rgb(z1, copas), extent=z1["ext"]); ax[2].set_title("Zoom 30 × 30 m")
ax[2].plot(cimas[:, 1] * z1["res"], cimas[:, 0] * z1["res"], "y+", ms=12, mew=2)
ventana(ax[2], z1, 60, 60, 30)
plt.tight_layout(); plt.show()

🔎 **Observa el zoom:** los polígonos parecen **más grandes** que las palmas que ves en la ortofoto y casi tocan a sus vecinas. Entre palma y palma el CHM marca 3–5 m aunque en la foto se ve arena. ¿Por qué? Piensa en las **frondas que cuelgan** y se traslapan, en cómo se **interpola** el CHM a partir de los puntos y en que la máscara solo corta a 2 m. Para algunas aplicaciones (p. ej. etiquetar imágenes, Cuaderno 2) conviene quedarse solo con la **parte alta de la copa**.

---
## Paso 6 · De píxeles a un inventario

Cada copa es ahora un **objeto** al que le podemos medir atributos con el CHM:

* **Altura máxima** (altura total del árbol) y **altura media** de la copa.
* **Área de copa** (m²) y **diámetro de copa equivalente**: $D = 2\sqrt{A/\pi}$
* **Posición** (coordenadas UTM de la cima).

Con la tabla de árboles calculamos **métricas de rodal**, que es lo que se reporta en un inventario.

In [ ]:
arboles = tabla_arboles(z1, copas, cimas)
arboles.head(10)

In [ ]:
#@title 📊 Métricas de rodal e histogramas
def metricas(arboles, zona, copas):
    n = len(arboles)
    top = arboles.altura_max_m.nlargest(max(1, int(0.2 * n)))
    return pd.Series(dtype=object, data={
        "Árboles detectados": n,
        "Superficie analizada (ha)": round(zona["area_ha"], 2),
        "Densidad (árboles/ha)": round(n / zona["area_ha"], 1),
        "Altura media (m)": round(arboles.altura_max_m.mean(), 2),
        "Altura dominante, 20 % más alto (m)": round(top.mean(), 2),
        "Diámetro de copa medio (m)": round(arboles.diam_copa_m.mean(), 2),
        "Cobertura de copas (%)": round(100 * (copas > 0).mean(), 1),
    })

display(metricas(arboles, z1, copas).to_frame("Zona 1 · Plantación"))

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].hist(arboles.altura_max_m, bins=25, color="#6b8e23", ec="k"); ax[0].set_title("Altura total (m)")
ax[1].hist(arboles.diam_copa_m, bins=25, color="#9BA075", ec="k"); ax[1].set_title("Diámetro de copa (m)")
ax[2].scatter(arboles.diam_copa_m, arboles.altura_max_m, s=12, c="#3D422A")
ax[2].set_xlabel("Diámetro de copa (m)"); ax[2].set_ylabel("Altura (m)"); ax[2].set_title("Relación altura – copa")
plt.tight_layout(); plt.show()

🔎 **Preguntas:**
* La densidad teórica de una plantación con marco de 8 × 8 m es 10 000 / 64 ≈ **156 árboles/ha**. ¿Qué densidad obtuviste? ¿Por qué podría diferir? (Mira la parte norte de la zona: ¿todo es plantación?)
* ¿Para qué te serviría la relación altura–copa en un inventario? (Pista: modelos alométricos para estimar diámetro normal, biomasa o carbono).

---
## Paso 7 · ¿Qué tan bueno es el conteo? Validación rápida

Un método automático **siempre** debe validarse contra una referencia (conteo en campo o fotointerpretación).

✏️ **Ejercicio 3** (3 min): en la imagen de abajo hay un cuadro de **30 × 30 m**. Cuenta **a ojo** las palmas cuyo centro cae dentro del cuadro, escribe tu número en `MI_CONTEO` y ejecuta la celda.

In [ ]:
#@title 👀 Conteo manual vs. automático
MI_CONTEO = 0  #@param {type:"integer"}
x0, y0, L = 90, 70, 30   # cuadro de validación en metros
fig, ax = plt.subplots(1, 2, figsize=(16, 8))
for a in ax:
    a.imshow(z1["rgb"], extent=z1["ext"])
    a.add_patch(plt.Rectangle((x0, y0), L, L, fill=False, ec="yellow", lw=3))
    ventana(a, z1, x0 - 5, y0 - 5, L + 10)
ax[0].set_title("Tu turno: cuenta las palmas dentro del cuadro")
dentro = arboles[(arboles.x_m >= x0) & (arboles.x_m < x0 + L) & (arboles.y_m >= y0) & (arboles.y_m < y0 + L)]
ax[1].plot(arboles.x_m, arboles.y_m, "r+", ms=14, mew=3)
ax[1].set_title(f"Algoritmo: {len(dentro)} árboles dentro del cuadro")
plt.tight_layout(); plt.show()
if MI_CONTEO > 0:
    err = 100 * (len(dentro) - MI_CONTEO) / MI_CONTEO
    print(f"Tu conteo: {MI_CONTEO} · Algoritmo: {len(dentro)} · Diferencia: {err:+.1f} %")
else:
    print("✏️ Escribe tu conteo en MI_CONTEO y vuelve a ejecutar.")

### El caso difícil: vegetación natural

Las plantaciones son el escenario ideal: árboles de una sola especie, misma edad, copas separadas y en cuadrícula. ¿Qué pasa en una **selva o vegetación ribereña**, con copas entrelazadas, varias alturas y especies?

✏️ **Ejercicio 4** (5 min): ejecuta el método en la **zona 2** con los mismos parámetros. Luego intenta mejorarlo moviendo los deslizadores. Anota qué combinación te parece más realista.

In [ ]:
#@title 🌳 Watershed en vegetación natural (zona 2)
SIGMA_M_Z2      = 0.4  #@param {type:"slider", min:0.1, max:2.0, step:0.1}
DIST_MIN_M_Z2   = 3.0  #@param {type:"slider", min:1.0, max:10.0, step:0.5}
ALTURA_MIN_M_Z2 = 2.0  #@param {type:"slider", min:1.0, max:8.0, step:0.5}

z2 = cargar_zona("zona2_natural")
suave2, cimas2, copas2 = segmentar(z2, SIGMA_M_Z2, DIST_MIN_M_Z2, ALTURA_MIN_M_Z2)
arboles2 = tabla_arboles(z2, copas2, cimas2)

fig, ax = plt.subplots(1, 3, figsize=(20, 7))
ax[0].imshow(z2["chm"], cmap="viridis", vmin=0, extent=z2["ext"]); ax[0].set_title("CHM zona 2")
ax[1].imshow(bordes_rgb(z2, copas2), extent=z2["ext"]); ax[1].set_title(f"{len(cimas2)} copas segmentadas")
ax[2].imshow(bordes_rgb(z2, copas2), extent=z2["ext"]); ax[2].set_title("Zoom 40 × 40 m")
ax[2].plot(cimas2[:, 1] * z2["res"], cimas2[:, 0] * z2["res"], "y+", ms=12, mew=2); ventana(ax[2], z2, 60, 60, 40)
plt.tight_layout(); plt.show()

display(pd.concat([metricas(arboles, z1, copas).to_frame("Zona 1 · Plantación"),
                   metricas(arboles2, z2, copas2).to_frame("Zona 2 · Natural")], axis=1))

🔎 **Discusión — limitaciones del watershed sobre CHM:**
* **Copas entrelazadas:** sin un "valle" claro entre árboles, el algoritmo los fusiona (subsegmentación).
* **Árboles con varias cimas** (latifoliadas, copas irregulares): se parten en varios (sobresegmentación).
* **Árboles suprimidos** bajo el dosel: el CHM solo "ve" la superficie superior; son **invisibles**.
* **No hay un parámetro universal:** σ y la distancia mínima dependen del ecosistema. Por eso en los proyectos de Kaab Map los guardamos como *presets* por tipo de vegetación y la limpieza final se hace en QGIS.

---
## Paso 8 · Exportar a QGIS

Guardamos las **copas** (polígonos) y las **cimas** (puntos) con todos sus atributos en un GeoPackage. Ábrelo en QGIS sobre la ortofoto para revisarlo y editarlo.

In [ ]:
#@title 💾 Exportar GeoPackage y descargar
pol, pts = a_geopackage(z1, copas, arboles, "inventario_zona1.gpkg")
print(f"inventario_zona1.gpkg → capa 'copas' ({len(pol)} polígonos) y capa 'cimas' ({len(pts)} puntos)")
print("Sistema de referencia:", z1["crs"])
if EN_COLAB:
    from google.colab import files
    files.download("inventario_zona1.gpkg")

---
## ✅ Resumen

1. El **CHM** (DSM − DTM) es la base: cada copa es una "montaña" de alturas.
2. **Suavizar (σ)** controla el ruido; **distancia mínima** y **altura mínima** definen qué es un árbol.
3. **Watershed con marcadores** inunda el CHM invertido desde cada cima y traza los límites de copa.
4. Del resultado salen **atributos por árbol** (altura, copa, posición) y **métricas de rodal** (densidad, cobertura, altura dominante).
5. Funciona muy bien en **plantaciones**; en **vegetación natural** requiere calibración y revisión manual.

➡️ En el **Cuaderno 2** veremos cómo detectar árboles **solo con la ortofoto** usando *Deep Learning*… y cómo el LiDAR puede ser el "maestro" que enseña a la red neuronal.